In [1]:
import multiprocessing as mp
import time
import os

In [2]:
mp.cpu_count()

12

In [3]:
mp.active_children()

[]

In [4]:
mp.get_all_start_methods()

['spawn', 'fork', 'forkserver']

In [5]:
mp.get_start_method()

'spawn'

In [7]:
os.getpid(), os.getppid()

(25969, 25244)

In [8]:
!ps -ef | grep 25969

  502 25969 25244   0  6:03   ??         0:00.96 /Library/Frameworks/Python.framework/Versions/3.14/Resources/Python.app/Contents/MacOS/Python -m ipykernel_launcher -f /Users/g.kandaurov/Library/Jupyter/runtime/kernel-6e89edbe-4bb6-431c-8464-e9ec86e7ea80.json
  502  6924 25969   0  7:01   ttys020    0:00.02 /bin/zsh -c ps -ef | grep 25969
  502  6933  6924   0  7:01   ttys020    0:00.00 grep 25969


In [9]:
def print_data(data):
    print(f"print_data {data=} -- start")

    time.sleep(10)

    print(f"print_data {data=} -- finish")


pr = mp.Process(target=print_data, args=("data_pr_1",), name="process_1")
pr.start()

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=93, pipe_handle=97)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'print_data'


In [10]:
ctx = mp.get_context("fork")

In [11]:
%%time

def print_data(data):
    print(f"print_data {data=} -- start")

    time.sleep(10)

    print(f"print_data {data=} -- finish")


pr = ctx.Process(target=print_data, args=("data_pr_1",), name="process_1")
print(f"after init  {pr.is_alive()=}, {pr.name=}, {pr.pid=}")
pr.start()
print(f"after start {pr.is_alive()=}, {pr.name=}, {pr.pid=}")
pr.join()
print(f"after join  {pr.is_alive()=}, {pr.name=}, {pr.pid=}")

after init  pr.is_alive()=False, pr.name='process_1', pr.pid=None
print_data data='data_pr_1' -- start
after start pr.is_alive()=True, pr.name='process_1', pr.pid=33765
print_data data='data_pr_1' -- finish
after join  pr.is_alive()=False, pr.name='process_1', pr.pid=33765
CPU times: user 6.81 ms, sys: 11.6 ms, total: 18.5 ms
Wall time: 10.1 s


In [12]:
!ps -ef | grep 33765

  502 35968 25969   0  7:08   ttys020    0:00.01 /bin/zsh -c ps -ef | grep 33765
  502 35970 35968   0  7:08   ttys020    0:00.00 grep 33765


In [13]:
import threading

In [14]:
%%time


N = 10 ** 9
N_THREADS = 10
N_JOBS = N // N_THREADS


def counter(a, b):
    print(f"name={threading.current_thread().name} {a=}, {b=} -- started")
    while a < b:
        a += 1
    print(f"name={threading.current_thread().name} -- finished")


def run():
    threads = [
        threading.Thread(
            target=counter,
            args=(i * N_JOBS, (i + 1) * N_JOBS),
            name=f"th_counter_{i}"
        )
        for i in range(N_THREADS)
    ]

    for th in threads:
        th.start()

    for th in threads:
        th.join()


run()

name=th_counter_0 a=0, b=100000000 -- started
name=th_counter_1 a=100000000, b=200000000 -- started
name=th_counter_2 a=200000000, b=300000000 -- started
name=th_counter_3 a=300000000, b=400000000 -- started
name=th_counter_4 a=400000000, b=500000000 -- started
name=th_counter_5 a=500000000, b=600000000 -- started
name=th_counter_6 a=600000000, b=700000000 -- started
name=th_counter_7 a=700000000, b=800000000 -- startedname=th_counter_8 a=800000000, b=900000000 -- started

name=th_counter_9 a=900000000, b=1000000000 -- started
name=th_counter_3 -- finished
name=th_counter_1 -- finished
name=th_counter_8 -- finished
name=th_counter_7 -- finished
name=th_counter_2 -- finished
name=th_counter_5 -- finished
name=th_counter_4 -- finished
name=th_counter_0 -- finished
name=th_counter_6 -- finished
name=th_counter_9 -- finished
CPU times: user 8.58 s, sys: 174 ms, total: 8.75 s
Wall time: 8.73 s


In [3]:
ctx = mp.get_context("fork")

In [4]:
%%time


N = 10 ** 7
N_PROCS = 10
N_JOBS = N // N_PROCS


def counter(a, b, lock, sem):
    with lock:
        print(f"pid={os.getpid()} {a=}, {b=} -- started")

    while a < b:
        with sem:
            a += 1

    with lock:
        print(f"pid={os.getpid()} -- finished")


def run():
    lock = ctx.Lock()
    sem = ctx.Semaphore(5)

    processes = [
        ctx.Process(
            target=counter,
            args=(i * N_JOBS, (i + 1) * N_JOBS, lock, sem),
            name=f"pr_counter_{i}"
        )
        for i in range(N_PROCS)
    ]

    for pr in processes:
        pr.start()

    for pr in processes:
        pr.join()


run()

pid=1876 a=0, b=1000000 -- started
pid=1877 a=1000000, b=2000000 -- started
pid=1878 a=2000000, b=3000000 -- started
pid=1879 a=3000000, b=4000000 -- started
pid=1881 a=4000000, b=5000000 -- started
pid=1882 a=5000000, b=6000000 -- started
pid=1883 a=6000000, b=7000000 -- started
pid=1884 a=7000000, b=8000000 -- started
pid=1885 a=8000000, b=9000000 -- started
pid=1886 a=9000000, b=10000000 -- started
pid=1876 -- finished
pid=1877 -- finished
pid=1878 -- finished
pid=1879 -- finished
pid=1883 -- finished
pid=1881 -- finished
pid=1882 -- finished
pid=1886 -- finished
pid=1884 -- finished
pid=1885 -- finished
CPU times: user 22.2 ms, sys: 49.8 ms, total: 72 ms
Wall time: 18.9 s


In [8]:
class User:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User[{self.name}, id={id(self)}]"


def update_data(data):
    print(f"update_data {data=} -- start {os.getpid()=}")

    data["to_update"] = "updated_val"
    data[(1, "2", (3,))] = "data"
    data["user"] = User("steve")

    print(f"update_data {data=} -- finish {os.getpid()=}")


data = {"to_update": "init_val", "fixed": 123}

update_data(data)

print(f"data after update", data)

update_data data={'to_update': 'init_val', 'fixed': 123} -- start os.getpid()=97660
update_data data={'to_update': 'updated_val', 'fixed': 123, (1, '2', (3,)): 'data', 'user': User[steve, id=4437152768]} -- finish os.getpid()=97660
data after update {'to_update': 'updated_val', 'fixed': 123, (1, '2', (3,)): 'data', 'user': User[steve, id=4437152768]}


In [9]:
class User:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User[{self.name}, id={id(self)}]"


def update_data(data):
    print(f"update_data {data=} -- start {os.getpid()=}")

    data["to_update"] = "updated_val"
    data[(1, "2", (3,))] = "data"
    data["user"] = User("steve")

    print(f"update_data {data=} -- finish {os.getpid()=}")


data = {"to_update": "init_val", "fixed": 123}

pr = ctx.Process(target=update_data, args=(data,))
pr.start()
pr.join()

print(f"data after update", data)

update_data data={'to_update': 'init_val', 'fixed': 123} -- start os.getpid()=42597
update_data data={'to_update': 'updated_val', 'fixed': 123, (1, '2', (3,)): 'data', 'user': User[steve, id=4436720144]} -- finish os.getpid()=42597
data after update {'to_update': 'init_val', 'fixed': 123}


In [17]:
class User:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"User[{self.name}, id={id(self)}]"


def update_data(data):
    print(f"update_data {data} -- start {os.getpid()=}")

    data["to_update"] = "updated_val"
    data[(1, "2", (3,))] = "data"
    data["user"] = User("steve")

    print(f"update_data {data} -- finish {os.getpid()=}")


data = {"to_update": "init_val", "fixed": 123}


with ctx.Manager() as manager:
    manager_data = manager.dict()
    for k, v in data.items():
        manager_data[k] = v

    pr = ctx.Process(target=update_data, args=(manager_data,))
    pr.start()
    pr.join()

    print(f"data after update {os.getpid()=}", manager_data)
    data = dict(manager_data)

print(f"data after manager {os.getpid()=}", data)

update_data {'to_update': 'init_val', 'fixed': 123} -- start os.getpid()=93317
update_data {'to_update': 'updated_val', 'fixed': 123, (1, '2', (3,)): 'data', 'user': User[steve, id=4469637200]} -- finish os.getpid()=93317
data after update os.getpid()=97660 {'to_update': 'updated_val', 'fixed': 123, (1, '2', (3,)): 'data', 'user': User[steve, id=4469637200]}
data after manager os.getpid()=97660 {'to_update': 'updated_val', 'fixed': 123, (1, '2', (3,)): 'data', 'user': User[steve, id=4450266352]}


In [12]:
os.getpid()

97660

In [19]:
%%time


def process_data(data, delay):
    print(f"pid={os.getpid()} {data=} -- started")

    time.sleep(delay)

    print(f"pid={os.getpid()} -- finished")


pr = ctx.Process(target=process_data, args=("data1", 40))
pr.start()

print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=98809, pr.is_alive()=True, pr.name='ForkProcess-25'
CPU times: user 1.72 ms, sys: 8.93 ms, total: 10.7 ms
Wall time: 9.86 ms
pid=98809 data='data1' -- started


In [20]:
print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=98809, pr.is_alive()=True, pr.name='ForkProcess-25'


In [21]:
!ps -ef | grep 98809

  502  5922 79134   0  7окт.26 ??         1:52.24 /Applications/Google Chrome.app/Contents/Frameworks/Google Chrome Framework.framework/Versions/152.0.7977.76/Helpers/Google Chrome Helper (Renderer).app/Contents/MacOS/Google Chrome Helper (Renderer) --type=renderer --origin-trial-disabled-features=CanvasTextNg|WebAssemblyCustomDescriptors --lang=ru --num-raster-threads=4 --enable-zero-copy --enable-gpu-memory-buffer-compositor-resources --enable-main-frame-before-activation --renderer-client-id=81788 --launch-time-ticks=3193464881391 --shared-files --metrics-shmem-handle=1752395122,r,28367598809817707,12623696764277879399,2097152 --field-trial-handle=1718379636,r,2433091897810679031,34384118164474065,262144 --variations-seed-version=20260904-090042.235000-production --pseudonymization-salt-handle=1935764596,r,4037422917381017458,13789707822457469063,4 --trace-process-track-uuid=3190785625090617506 --seatbelt-client=346
  502 98809 97660   0  8:21   ??         0:00.01 /Library/Framework

In [29]:
!kill -9 98809

zsh:kill:1: kill 98809 failed: no such process


In [23]:
!ps -ef | grep 98809

  502  5922 79134   0  7окт.26 ??         1:52.24 /Applications/Google Chrome.app/Contents/Frameworks/Google Chrome Framework.framework/Versions/152.0.7977.76/Helpers/Google Chrome Helper (Renderer).app/Contents/MacOS/Google Chrome Helper (Renderer) --type=renderer --origin-trial-disabled-features=CanvasTextNg|WebAssemblyCustomDescriptors --lang=ru --num-raster-threads=4 --enable-zero-copy --enable-gpu-memory-buffer-compositor-resources --enable-main-frame-before-activation --renderer-client-id=81788 --launch-time-ticks=3193464881391 --shared-files --metrics-shmem-handle=1752395122,r,28367598809817707,12623696764277879399,2097152 --field-trial-handle=1718379636,r,2433091897810679031,34384118164474065,262144 --variations-seed-version=20260904-090042.235000-production --pseudonymization-salt-handle=1935764596,r,4037422917381017458,13789707822457469063,4 --trace-process-track-uuid=3190785625090617506 --seatbelt-client=346
  502 98809 97660   0  8:21   ??         0:00.00 <defunct>
  502  2

In [30]:
!ps -ef | grep 98809

  502  5922 79134   0  7окт.26 ??         1:52.26 /Applications/Google Chrome.app/Contents/Frameworks/Google Chrome Framework.framework/Versions/152.0.7977.76/Helpers/Google Chrome Helper (Renderer).app/Contents/MacOS/Google Chrome Helper (Renderer) --type=renderer --origin-trial-disabled-features=CanvasTextNg|WebAssemblyCustomDescriptors --lang=ru --num-raster-threads=4 --enable-zero-copy --enable-gpu-memory-buffer-compositor-resources --enable-main-frame-before-activation --renderer-client-id=81788 --launch-time-ticks=3193464881391 --shared-files --metrics-shmem-handle=1752395122,r,28367598809817707,12623696764277879399,2097152 --field-trial-handle=1718379636,r,2433091897810679031,34384118164474065,262144 --variations-seed-version=20260904-090042.235000-production --pseudonymization-salt-handle=1935764596,r,4037422917381017458,13789707822457469063,4 --trace-process-track-uuid=3190785625090617506 --seatbelt-client=346
  502 12465 97660   0  8:24   ttys020    0:00.02 /bin/zsh -c ps -ef

In [28]:
print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=98809, pr.is_alive()=False, pr.name='ForkProcess-25'


In [31]:
%%time


def process_data(data, delay):
    print(f"pid={os.getpid()} {data=} -- started")

    time.sleep(delay)

    print(f"pid={os.getpid()} -- finished")


pr = ctx.Process(target=process_data, args=("data1", 40))
pr.start()

print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=14985, pr.is_alive()=True, pr.name='ForkProcess-26'
CPU times: user 1.72 ms, sys: 7.21 ms, total: 8.93 ms
Wall time: 8.06 ms
pid=14985 data='data1' -- started


In [32]:
print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=14985, pr.is_alive()=True, pr.name='ForkProcess-26'


In [33]:
!ps -ef | grep 14985

  502 14985 97660   0  8:24   ??         0:00.01 /Library/Frameworks/Python.framework/Versions/3.14/Resources/Python.app/Contents/MacOS/Python -m ipykernel_launcher -f /Users/g.kandaurov/Library/Jupyter/runtime/kernel-6e89edbe-4bb6-431c-8464-e9ec86e7ea80.json
  502 15814 97660   0  8:24   ttys020    0:00.02 /bin/zsh -c ps -ef | grep 14985
  502 15816 15814   0  8:24   ttys020    0:00.00 grep 14985


In [34]:
!kill -15 14985

In [35]:
print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=14985, pr.is_alive()=False, pr.name='ForkProcess-26'


In [36]:
!ps -ef | grep 14985

  502 18245 97660   0  8:25   ttys020    0:00.02 /bin/zsh -c ps -ef | grep 14985
  502 18247 18245   0  8:25   ttys020    0:00.00 grep 14985


In [37]:
import signal

In [44]:
%%time


def process_data(data, delay):
    print(f"pid={os.getpid()} {data=} -- started")

    def term_handler(*args, **kwargs):
        print(f"term_handler {args=}, {kwargs=}")        

    signal.signal(signal.SIGTERM, term_handler)

    time.sleep(delay)

    print(f"pid={os.getpid()} -- finished")


pr = ctx.Process(target=process_data, args=("data1", 5))
pr.start()

print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=37732, pr.is_alive()=True, pr.name='ForkProcess-28'
CPU times: user 1.73 ms, sys: 6.82 ms, total: 8.55 ms
Wall time: 7.31 ms
pid=37732 data='data1' -- started
pid=37732 -- finished


In [45]:
print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=37732, pr.is_alive()=True, pr.name='ForkProcess-28'


In [46]:
%%time


def process_data(data, delay):
    print(f"pid={os.getpid()} {data=} -- started")

    def term_handler(*args, **kwargs):
        print(f"term_handler {args=}, {kwargs=}")        

    signal.signal(signal.SIGTERM, term_handler)

    time.sleep(delay)

    print(f"pid={os.getpid()} -- finished")


pr = ctx.Process(target=process_data, args=("data1", 50))
pr.start()

print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=39359, pr.is_alive()=True, pr.name='ForkProcess-29'
CPU times: user 1.94 ms, sys: 7.67 ms, total: 9.6 ms
Wall time: 8.45 ms
pid=39359 data='data1' -- started
term_handler args=(15, <frame at 0x106c41e40, file '<timed exec>', line 9, code process_data>), kwargs={}
pid=39359 -- finished


In [47]:
print(f"{pr.pid=}, {pr.is_alive()=}, {pr.name=}")

pr.pid=39359, pr.is_alive()=True, pr.name='ForkProcess-29'


In [49]:
!kill -15 39359

In [50]:
!kill -l

HUP INT QUIT ILL TRAP ABRT EMT FPE KILL BUS SEGV SYS PIPE ALRM TERM URG STOP TSTP CONT CHLD TTIN TTOU IO XCPU XFSZ VTALRM PROF WINCH INFO USR1 USR2


In [51]:
import subprocess

In [60]:
resp = subprocess.run(["ls", "-al"], capture_output=True, text=True)

In [62]:
resp

CompletedProcess(args=['ls', '-al'], returncode=0, stdout='total 1472\ndrwxr-xr-x   6 g.kandaurov  staff     192 16 окт.  20:35 \x1b\x1b.\x1b[m\x1b[m\ndrwxr-xr-x  18 g.kandaurov  staff     576 16 окт.  16:46 \x1b\x1b..\x1b[m\x1b[m\ndrwxr-xr-x   3 g.kandaurov  staff      96 16 окт.  18:03 \x1b\x1b.ipynb_checkpoints\x1b[m\x1b[m\n-rw-r--r--   1 g.kandaurov  staff   38933 16 окт.  20:35 class_07.ipynb\n-rwxr-xr-x@  1 g.kandaurov  staff    2026 16 окт.  17:08 \x1b\x1bhomework.md\x1b[m\x1b[m\n-rw-r--r--@  1 g.kandaurov  staff  706452 16 окт.  17:04 lesson-07.pdf\n', stderr='')

In [67]:
resp.stdout.split("\n")

['total 1472',
 'drwxr-xr-x   6 g.kandaurov  staff     192 16 окт.  20:35 \x1b\x1b.\x1b[m\x1b[m',
 'drwxr-xr-x  18 g.kandaurov  staff     576 16 окт.  16:46 \x1b\x1b..\x1b[m\x1b[m',
 'drwxr-xr-x   3 g.kandaurov  staff      96 16 окт.  18:03 \x1b\x1b.ipynb_checkpoints\x1b[m\x1b[m',
 '-rw-r--r--   1 g.kandaurov  staff   38933 16 окт.  20:35 class_07.ipynb',
 '-rwxr-xr-x@  1 g.kandaurov  staff    2026 16 окт.  17:08 \x1b\x1bhomework.md\x1b[m\x1b[m',
 '-rw-r--r--@  1 g.kandaurov  staff  706452 16 окт.  17:04 lesson-07.pdf',
 '']

In [61]:
resp.returncode

0

In [58]:
resp = subprocess.run(["false"], capture_output=True, text=True)

In [59]:
resp.returncode

1

In [68]:
resp = subprocess.run(["python", "-c", "print(123)"], capture_output=True)

In [69]:
resp

CompletedProcess(args=['python', '-c', 'print(123)'], returncode=0, stdout=b'123\n', stderr=b'')